# ニューラル機械翻訳: Keras 3 による RNN Encoder-Decoder の構築

**学習目標**
1.  seq2seq タスク用の効率的な `tf.data.Dataset` パイプラインを構築する方法を学ぶ。
2.  `keras.layers.TextVectorization` レイヤを使用してテキストを前処理する方法を学ぶ。
3.  Functional API を使用して Keras で Encoder-Decoder モデルをトレーニングする方法を学ぶ。
4.  推論用の個別のエンコーダ モデルとデコーダ モデルを作成する方法を学ぶ。
5.  ゼロから翻訳（デコード）関数を実装する方法を学ぶ。
6.  BLEU スコアを使用して翻訳モデルを評価する方法を学ぶ。


***注: 実行を高速化するために、GPU ランタイムを使用していることを確認してください。このノートブックには NVIDIA T4 GPU が推奨されます。***

## はじめに

このノートブックでは、Keras 3 で最新の RNN Encoder-Decoder アーキテクチャを使用して、スペイン語から英語への翻訳モデルを構築します。

まず、`tf.data.Dataset` API を使用して効率的でスケーラブルな入力パイプラインを構築します。ワークフローの重要な部分は、標準化やトークン化から整数エンコーディングに至るすべてのテキスト前処理を、モデル内で直接処理するために `TextVectorization` レイヤを使用することです。

次に、Keras Functional API を使用して RNN Encoder-Decoder モデルを構築し、トレーニングします。トレーニング後、トレーニング済みモデルのレイヤから専用のエンコーダとデコーダという 2 つの特化モデルを作成します。これらの特化モデルは推論を実行するために不可欠であり、単語ごとに翻訳を生成する関数を実装できるようになります。

最後に、業界標準の BLEU スコアを使用してモデルの翻訳の品質を評価します。

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:
import pickle
import re
import string
import sys

import evaluate
import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from keras.layers import GRU, Dense, Embedding, Input
from keras.models import Model, load_model
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
SEED = 42
MODEL_PATH = "translate_models/baseline"
DATA_URL = (
    "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
)
LOAD_CHECKPOINT = False

In [ ]:
keras.utils.set_random_seed(SEED)

## データのダウンロード

http://www.manythings.org/anki/ から提供されている言語データセットを使用します。このデータセットには、次の形式のスペイン語-英語の翻訳ペアが含まれています。

```
May I borrow this book?	¿Puedo tomar prestado este libro?
```

このデータセットは、ネイティブ スピーカーによるコミュニティ投稿の翻訳プラットフォームである http://tatoeba.org/ から収集された、厳選された 12 万件の翻訳ペアです。

In [ ]:
path_to_zip = keras.utils.get_file("spa-eng.zip", origin=DATA_URL, extract=True)

path_to_file = os.path.join(
    os.path.dirname(path_to_zip), "spa-eng_extracted/spa-eng/spa.txt"
)
print("Translation data stored at:", path_to_file)

In [ ]:
data = pd.read_csv(
    path_to_file, sep="\t", header=None, names=["english", "spanish"]
)

In [ ]:
data.sample(3)

## tf.data パイプラインの作成
まず、TensorFlow で効率的かつスケーラブルな入力パイプラインを構築するための標準である `tf.data` API を使用して、トレーニング用および評価用データセットを構築します。このアプローチにより、メモリ効率の高い方法でデータを処理し、Keras とシームレスに統合できます。

プロセスは次のとおりです。

1. データの読み込み: `tf.data.Dataset.from_tensor_slices` を使用して、pandas DataFrame から直接 `tf.data.Dataset` を作成します。これにより、各要素が（スペイン語、英語）の文のペアであるデータセットが作成されます。

2. データの分割: `.take()` メソッドと `.skip()` メソッドを使用して、トレーニング セットと検証セットを作成します。これは、すべてを一度にメモリに読み込むことなく、データをクリーンかつ効率的に分割する方法です。

3. 標準化関数の定義: 生のテキストを前処理するための `custom_standardization` 関数を作成します。この関数は、テキストの小文字化、句読点の周囲へのスペースの追加、そして最も重要な各文への `<start>` および `<end>` トークンの追加を行うことで、元の `preprocess_sentence` 関数のロジックを再現します。これは `tf.strings` 操作を使用して構築されているため、`TextVectorization` レイヤに直接組み込んで GPU / TPU 上で効率的に実行できます。

In [ ]:
# 定数の定義
BUFFER_SIZE = 32000
BATCH_SIZE = 64
TEST_PROP = 0.2
NUM_EXAMPLES = 30000

# pandas DataFrame から単一のデータセットを作成
full_dataset = tf.data.Dataset.from_tensor_slices(
    (data["spanish"][:NUM_EXAMPLES], data["english"][:NUM_EXAMPLES])
)

# take() と skip() を使用してトレーニング分割と検証分割を作成
TRAIN_SIZE = int(NUM_EXAMPLES * (1 - TEST_PROP))
train_raw = full_dataset.take(TRAIN_SIZE)
val_raw = full_dataset.skip(TRAIN_SIZE)

# TextVectorization 用のカスタム標準化関数を定義


@keras.utils.register_keras_serializable()
def custom_standardization(input_string):
    # 小文字化し、前後の空白文字を削除
    s = tf.strings.lower(input_string)
    s = tf.strings.strip(s)

    # 句読点の周囲にスペースを追加
    s = tf.strings.regex_replace(s, r"([?.!,¿])", r" \1 ")

    # 連続する複数のスペースを単一のスペースに置換
    s = tf.strings.regex_replace(s, r'[" "]+', " ")

    # 不要な文字をスペースに置き換えて除外
    s = tf.strings.regex_replace(s, r"[^a-zA-Z?.!,¿]+", " ")

    # クリーニングによって生じた前後のスペースを再度削除
    s = tf.strings.strip(s)

    # <start> および <end> トークンを追加
    s = tf.strings.join(["<start>", s, "<end>"], separator=" ")
    return s

### TextVectorization レイヤの作成と適応（Adapt）
生のテキスト データセットの準備ができたので、文を文字列からモデルが理解できる整数シーケンスに変換する必要があります。Keras 3 でこれを効率的かつ最新の方法で行うには、`keras.layers.TextVectorization` レイヤを使用します。このレイヤは、モデルのグラフ内で直接トークン化、語彙の作成、整数エンコーディングを処理します。

ソース言語（スペイン語）用とターゲット言語（英語）用の 2 つの独立した TextVectorization レイヤを作成します。トークン化の前に特定のルールに従ってテキストが前処理される（`<start>` および `<end>` トークンの追加を含む）ように、`custom_standardization` 関数を `standardize` 引数に渡します。

最も重要なステップは、`.adapt()` メソッドの呼び出しです。`adapt` メソッドはトレーニング データセットからテキストを読み取り、すべての一意の単語の語彙を構築します。レイヤが各単語から一意の整数インデックスへのマッピングを学習するのはこのステップです。ソースとターゲットの両方のベクトル化レイヤに対して、それぞれのテキスト データでこれを行います。

***注: 前処理レイヤをデータセットに適応させるため、このセルの実行には 10〜15 分ほどかかります***

In [ ]:
# TextVectorization レイヤの作成と適応（adapt）
source_vectorization = keras.layers.TextVectorization(
    max_tokens=None,
    output_mode="int",
    output_sequence_length=None,
    standardize=custom_standardization,
)

target_vectorization = keras.layers.TextVectorization(
    max_tokens=None,
    output_mode="int",
    output_sequence_length=None,
    standardize=custom_standardization,
)

# レイヤをトレーニング データに適応させる
source_text = train_raw.map(lambda x, y: x)
target_text = train_raw.map(lambda x, y: y)

source_vectorization.adapt(source_text)
target_vectorization.adapt(target_text)

### 最終的なデータセットの作成
TextVectorization レイヤが語彙を学習したので、最終的な入力パイプラインを構築できます。

まず、モデルの Embedding レイヤで後で使用できるように語彙サイズを保存します。次に、2 つのヘルパー関数を定義します。

1. `vectorize_text`: 生のテキスト文を受け取り、適切な TextVectorization レイヤを適用して整数シーケンスに変換する関数です。

2. `create_dataset`: Encoder-Decoder モデル用に整数化されたシーケンスを準備する関数です。デコーダはトレーニング中にターゲット シーケンスの 2 つのバージョンを必要とします。1 つは入力用（次の単語を予測するため）、もう 1 つは出力用（損失計算のために予測と比較するため）です。この関数は以下を作成します。
   - target_in: 最後のトークンが削除されたターゲット シーケンス。
   - target_out: 最初のトークン（`<start>`）が削除されたターゲット シーケンス。

最後に、いくつかの `tf.data` メソッドを連結して、最終的な `train_dataset` と `eval_dataset` を構築します。トレーニング データを `.shuffle()` し、両方のデータセットを `.batch()` 化し、`.map()` を使用して前処理関数を適用し、パフォーマンス向上のために `.prefetch()` を呼び出します。また、最初のバッチの形状を検査して、パイプラインが正しく設定されていることを確認します。

In [ ]:
# 語彙サイズの設定
INPUT_VOCAB_SIZE = source_vectorization.vocabulary_size()
TARGET_VOCAB_SIZE = target_vectorization.vocabulary_size()


def vectorize_text(source, target):
    source = source_vectorization(source)
    target = target_vectorization(target)
    return source, target


def create_dataset(source, target):
    target_in = target[:, :-1]
    target_out = target[:, 1:]
    return (source, target_in), target_out


# 最終的なトレーニング用および検証用データセットの作成
train_dataset = (
    train_raw.shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
    .map(vectorize_text)
    .map(create_dataset)
    .prefetch(tf.data.AUTOTUNE)
)

eval_dataset = (
    val_raw.batch(BATCH_SIZE)
    .map(vectorize_text)
    .map(create_dataset)
    .prefetch(tf.data.AUTOTUNE)
)

# Embedding レイヤの最大長を設定
# データセットの element_spec を調べることでこれを行います
for (source, target_in), target_out in train_dataset.take(1):
    max_length_inp = source.shape[1]
    max_length_targ = target_in.shape[1]
    print("Source shape:", source.shape)
    print("Target in shape:", target_in.shape)
    print("Target out shape:", target_out.shape)

### データのバッチのプレビュー
データ パイプラインが正しく機能していることを確認するために、`train_dataset` から 1 つのバッチを取得し、最初のサンプルを確認してみましょう。整数テンソルをテキストに戻して、トレーニング中にモデルが何を受け取るかを確認します。

In [ ]:
# ベクトル化レイヤから語彙を取得
source_vocab = source_vectorization.get_vocabulary()
source_index_lookup = {i: word for i, word in enumerate(source_vocab)}

target_vocab = target_vectorization.get_vocabulary()
target_index_lookup = {i: word for i, word in enumerate(target_vocab)}

# トークン ID のテンソルを文字列に戻すヘルパー関数


def to_text(tensor, lookup_dict):
    # パディング トークン（ID 0）を除外して単語を結合
    return " ".join([lookup_dict[i] for i in tensor.numpy() if i != 0])


# トレーニング データセットから 1 バッチ取得し、最初のサンプルを確認
for (source_batch, target_in_batch), target_out_batch in train_dataset.take(1):
    # バッチから最初のサンプルを取得
    source_example = source_batch[0]
    target_in_example = target_in_batch[0]
    target_out_example = target_out_batch[0]

    print("--- Example from a Training Batch ---")
    print("\nSource (Spanish):")
    print(f"  Tensor: {source_example.numpy()}")
    print(f"  Text:   {to_text(source_example, source_index_lookup)}")

    print("\nTarget Input (English - for Decoder):")
    print(f"  Tensor: {target_in_example.numpy()}")
    print(f"  Text:   {to_text(target_in_example, target_index_lookup)}")

    print("\nTarget Output (English - for Loss Calculation):")
    print(f"  Tensor: {target_out_example.numpy()}")
    print(f"  Text:   {to_text(target_out_example, target_index_lookup)}")

## RNN Encoder-Decoder モデルのトレーニング

Encoder-Decoder アーキテクチャを使用しますが、RNN に入力する前に単語を潜在空間に埋め込みます。

In [ ]:
EMBEDDING_DIM = 256
HIDDEN_UNITS = 1024

### エンコーダの構築

**演習: エンコーダ モデルの構築**  
標準的な Encoder-Decoder 翻訳機用のエンコーダ コンポーネントを構築します。エンコーダの役割は、入力文全体を処理し、その意味を単一の固定サイズの「思考ベクトル（thought vector）」（または「コンテキスト ベクトル」）に圧縮することです。このベクトルは最終的にデコーダに渡されます。

データの流れを明確に定義できる Keras Functional API を使用してエンコーダを構築します。エンコーダには主に次の 2 つのレイヤがあります。

1. **Embedding レイヤ**: 整数エンコードされた語彙を受け取り、各単語の密ベクトル表現（埋め込み）を学習します。これらの埋め込みは、単語間の意味的関係を捉えることができます。

2. **GRU（Gated Recurrent Unit）レイヤ**: 単語埋め込みのシーケンスを 1 つずつ処理するリカレント ニューラル ネットワーク（RNN）の一種です。入力文全体を処理した後の GRU の最終隠れ状態を取得するために、`return_state=True` で設定します。この最終隠れ状態が入力文の意味を凝縮した「思考ベクトル」であり、デコーダに渡されます。このレイヤを GPU と互換性を持たせるために、引数 `reset_after=False` を追加してください。詳細についてはこちらの[ドキュメント](https://keras.io/api/layers/recurrent_layers/gru/)を参照してください。

In [ ]:
encoder_inputs = Input(shape=(None,), name="encoder_input")

# TODO - Embedding レイヤを定義
encoder_inputs_embedded = Embedding()(encoder_inputs)

encoder_rnn = GRU()  # TODO - GRU レイヤを定義

encoder_outputs, encoder_state = encoder_rnn(encoder_inputs_embedded)

### デコーダの構築
**演習: デコーダの定義**  
エンコーダからの「思考ベクトル」を受け取り、翻訳された文を単語ごとに生成するようにデコーダを構築します。

そのアーキテクチャはエンコーダと対称的であり、Embedding レイヤと GRU レイヤを含みます。決定的な違いは、デコーダの GRU レイヤをエンコーダの最終隠れ状態で初期化することです（`initial_state=encoder_state`）。このようにして、デコーダは元の文からのコンテキストを受け取り、それを使用して正しい翻訳を生成します。

In [ ]:
decoder_inputs = Input(shape=(None,), name="decoder_input")

# TODO - Embedding レイヤを定義
decoder_inputs_embedded = Embedding()(decoder_inputs)

decoder_rnn = GRU()  # TODO - Decoder レイヤを定義

decoder_outputs, decoder_state = decoder_rnn(
    # TODO - "initial_state" パラメータを使用してエンコーダとデコーダを接続
)

Encoder-Decoder アーキテクチャの最後の部分は、`decoder_output` から次の単語の確率ベクトル、つまり次の単語の `predictions` を作成する softmax `Dense` レイヤです。

In [ ]:
decoder_dense = Dense(TARGET_VOCAB_SIZE, activation="softmax", name="dense")

predictions = decoder_dense(decoder_outputs)

### モデルの作成とコンパイル

**演習: モデルのコンパイル コードを完成させる**
トレーニング可能なモデルのコンパイル コードを完成させ、Model オブジェクトをインスタンス化して、エンコーダとデコーダの入力、および出力としての最終的な予測を指定します。

トレーニング用にモデルを構成してコンパイルします。`adam` オプティマイザと `sparse_categorical_crossentropy` 損失関数を使用します。ターゲットが整数（単語のインデックス）であり、モデルの出力が語彙全体の確率分布であるため、この損失関数は今回のタスクに最適です。

最後に、`.summary()` を呼び出して、レイヤ、出力の形状、トレーニング可能なパラメータ数など、モデルのアーキテクチャの有用な概要を出力します。

In [ ]:
model = Model(inputs=[encoder_inputs, decoder_inputs], outputs=predictions)

# TODO - パラメータを入力
model.compile()
# TODO - モデルの概要を出力

モデルをトレーニングします。

***注: 十分な翻訳パフォーマンスを得るには、`EPOCHS` の数を 10〜15 に更新してください。ただし、トレーニング時間が長くなります。***

In [ ]:
EPOCHS = 5

history = model.fit(
    train_dataset, validation_data=eval_dataset, epochs=EPOCHS, verbose=2
)

### 翻訳（推論）関数の実装

モデルのトレーニングが完了したので、翻訳に使用する方法が必要です。モデルは入力としてソース文（スペイン語）とターゲット文（英語）の両方を想定しているため、スペイン語の文に対して単純に `model.predict()` を呼び出すことはできません。トレーニング中には、「Teacher Forcing」と呼ばれる手法で正解のターゲット文を使用しました。しかし、推論時にはターゲット文はありません。ターゲット文こそが生成すべきものだからです。

解決策は、ループ内で単語ごとに翻訳を生成することです。プロセスは次のようになります。

1.  入力文（例: "No estamos comiendo."）を取得し、エンコーダに通して最終隠れ状態（「思考ベクトル」）を取得します。
2.  `<start>` トークンでデコーダを開始します。
3.  `<start>` トークンとエンコーダの状態をデコーダに入力して、翻訳の最初の単語を予測します。
4.  予測された単語を取得し、次のステップの入力としてデコーダにフィードバックし、デコーダの新しい状態を使用します。
5.  翻訳の完了を示す `<end>` トークンをデコーダが予測するまで、このプロセスを繰り返します。

**演習**  
これを実装するために、トレーニングしたばかりのモデルのレイヤを使用して、2 つの新しい特化モデルを作成します。

*   生の文字列の文を受け取り、ベクトル化して、エンコーダの最終隠れ状態を返す **`encoder_model`**。
*   現在予測されているシーケンスと隠れ状態を受け取り、次の単語の予測を更新された隠れ状態とともに返す **`decoder_model`**。

In [ ]:
# --- 推論用エンコーダ モデルの作成 ---
# このモデルは生のスペイン語の文をエンコーダの最終状態に変換します。

# 生の文字列用の入力レイヤ
encoder_string_input = keras.Input(
    shape=(1,), dtype="string", name="encoder_string_input"
)

# 適応済みレイヤを使用して文字列をベクトル化
# TODO

# トレーニング済みの Embedding レイヤを再利用
# TODO

# トレーニング済みの GRU レイヤを再利用
# TODO

# 推論用の最終的なエンコーダ モデルを作成
encoder_model = Model(
    inputs=encoder_string_input,
    outputs=encoder_state_output,
    name="inference_encoder",
)

In [ ]:
# --- 推論用デコーダ モデルの作成 ---
# このモデルの構造はトークン ID を処理するため、元のモデルと似ています。

# デコーダ用の入力レイヤ
decoder_input = keras.Input(shape=(None,), name="decoder_input")
decoder_state_input = keras.Input(
    shape=(HIDDEN_UNITS,), name="decoder_state_input"
)

# メインモデルからトレーニング済みレイヤを再利用
# TODO

# デコーダ グラフを構築
# TODO

# 推論用の最終的なデコーダ モデルを作成
decoder_model = Model(
    inputs=[decoder_input, decoder_state_input],
    outputs=[decoder_predictions, decoder_state_output],
    name="inference_decoder",
)

### 翻訳の実行

特化されたエンコーダ モデルとデコーダ モデルの準備ができたので、すべてを結び付けて翻訳を実行するメインの `decode_sequences` 関数を定義できます。

この関数は、ステップバイステップのデコード プロセスを実装します。

1.  **入力のエンコード**: 生の入力文が `encoder_model` に渡され、初期の「思考ベクトル」または隠れ状態が取得されます。

2.  **シーケンスの初期化**: バッチ内の各文について、`<start>` トークンのみを含むターゲット シーケンスが作成されます。

3.  **反復デコード**: ループ内で、現在のターゲット シーケンスと隠れ状態を指定して `decoder_model` を呼び出し、次のトークンを予測します。簡単のため、最も確率の高いトークンを予測として選択するために `argmax` を使用します。

4.  **追加と更新**: 予測されたトークンが結果に追加され、新しいトークンとデコーダから更新された隠れ状態を使用してプロセスが繰り返されます。

5.  **停止条件**: 特別な `<end>` トークンが生成されるか、最大シーケンス長に達するまでループが継続します。

最後に、いくつかのサンプル文で関数をテストし、実際の翻訳結果を確認します。

In [ ]:
# --- 翻訳関数（decode_sequences）の定義 ---

# 語彙を取得し、逆マッピングを作成
targ_vocab = target_vectorization.get_vocabulary()
targ_index_lookup = dict(zip(range(len(targ_vocab)), targ_vocab))
start_token_id = target_vectorization.get_vocabulary().index("<start>")
end_token_id = target_vectorization.get_vocabulary().index("<end>")


def decode_sequences(input_sentences, max_decode_length=50):
    """
    Arguments:
        input_sentences: A tensor of raw strings in the source language.
    Returns:
        A list of translated sentences.
    """
    batch_size = input_sentences.shape[0]

    # 入力文字列をエンコードして初期状態を取得。
    states_value = encoder_model(input_sentences)

    # ターゲット シーケンスを <start> トークン ID で初期化。
    target_seq = keras.ops.full((batch_size, 1), start_token_id, dtype="int32")

    decoded_sentences = ["" for _ in range(batch_size)]

    for i in range(max_decode_length):
        output_tokens, decoder_state = decoder_model([target_seq, states_value])

        # トークンをサンプリング（簡単のため argmax を使用）
        sampled_token_index = keras.ops.argmax(output_tokens[:, -1, :], axis=-1)

        # 次のイテレーションのためにターゲット シーケンスを更新
        target_seq = keras.ops.expand_dims(sampled_token_index, axis=-1)

        # デコーダの状態を更新
        states_value = decoder_state

        for j in range(batch_size):
            token = targ_index_lookup[sampled_token_index[j].numpy()]
            if token == "<end>":
                continue  # end トークンを出力に追加しない
            decoded_sentences[j] += token + " "

    return [s.strip() for s in decoded_sentences]


sentences = [
    "No estamos comiendo.",
    "Está llegando el invierno.",
]

machine_translations = decode_sequences(keras.ops.convert_to_tensor(sentences))
print(machine_translations)

In [ ]:
def create_dataset(encoder_input, decoder_input):
    
    # 1 つ前にシフト
    target = tf.roll(decoder_input, -1, 1)

    # 最後の列を 0 に置き換え
    zeros = tf.zeros([target.shape[0], 1], dtype=tf.int32)
    target = tf.concat((target[:, :-1], zeros), axis=-1)

    dataset = # TODO

    return dataset

上記の関数を使用して、実際のトレーニング用および評価用データセットを作成しましょう。

In [ ]:
BUFFER_SIZE = len(input_tensor_train)
BATCH_SIZE = 64

In [ ]:
train_dataset = (
    create_dataset(input_tensor_train, target_tensor_train)
    .shuffle(BUFFER_SIZE)
    .repeat()
    .batch(BATCH_SIZE, drop_remainder=True)
)


eval_dataset = create_dataset(input_tensor_val, target_tensor_val).batch(
    BATCH_SIZE, drop_remainder=True
)

これで予測の準備が整いました。

In [ ]:
sentences = [
    "No estamos comiendo.",
    "Está llegando el invierno.",
    "El invierno se acerca.",
    "Tom no comio nada.",
    "Su pierna mala le impidió ganar la carrera.",
    "Su respuesta es erronea.",
    "¿Qué tal si damos un paseo después del almuerzo?",
]

reference_translations = [
    "We're not eating.",
    "Winter is coming.",
    "Winter is coming.",
    "Tom ate nothing.",
    "His bad leg prevented him from winning the race.",
    "Your answer is wrong.",
    "How about going for a walk after lunch?",
]

machine_translations = decode_sequences(keras.ops.convert_to_tensor(sentences))

for i in range(len(sentences)):
    print("-")
    print("INPUT:")
    print(sentences[i])
    print("REFERENCE TRANSLATION:")
    print(reference_translations[i])
    print("MACHINE TRANSLATION:")
    print(machine_translations[i])

## 評価指標（BLEU）

画像分類などとは異なり、機械翻訳には唯一の正解というものが存在しません。しかし、現在の損失指標であるクロス エントロピーは、機械翻訳が参照翻訳とまったく同じ単語をまったく同じ順序で一致させた場合にのみ評価します。

自然言語の評価のためのより優れた指標を開発するために、多くの試みが行われてきました。現在最も一般的なものは、BLEU（Bilingual Evaluation Understudy）です。

- 計算が迅速かつ低コストである。
- 単語やフレーズの順序に柔軟性を持たせることができる。
- 理解しやすい。
- 言語に依存しない。
- 人間による評価と高い相関関係がある。
- 広く採用されている。

スコアは 0 から 1 の範囲で、1 は完全一致を表します。

順序に関係なく、機械翻訳と参照テキストの間で一致する n-gram をカウントすることによって機能します。BLEU-4 は 1〜4（1-gram、2-gram、3-gram、4-gram）の一致する n-gram をカウントします。通常、BLEU-1 と BLEU-4 の両方を報告します。

同義語が考慮されないため依然として不完全であり、実行可能であれば人間による評価が依然として最良です。ただし、BLEU は自動指標としての選択肢の中で最も優れていると一般に考えられています。

ここでは、Hugging Face の `evaluate` フレームワークの実装を使用します。

トレーニング中には正しいデコーダ入力が使用されるため、トレーニング中に BLEU を計算することはできません。代わりに、今ここで計算します。

詳細情報: https://machinelearningmastery.com/calculate-bleu-score-for-text-python/

In [ ]:
def postprocess(sentence):
    filtered = list(filter(lambda x: x != "" and x != "<end>", sentence))
    return " ".join(filtered)

評価セット内のすべての文のペアに対して、`bleu_1` スコアと `bleu_4` スコアを平均化してみましょう。次のセルの実行には約 1 分（データセット全体の評価では約 8 分）かかりますが、その大部分は検証セット内の文のデコードです。完了するまでお待ちください。

In [ ]:
NUM_EVALUATE = 1000

source_sentences = []
reference = []  # これは BLEU スコア計算用のリストのリストになります

# 生の検証データセットを反復処理して、ソース テキストと参照テキストを取得します。
# データセットからの EagerTensor を文字列に変換するために .numpy().decode() を使用します。
for spa, eng in tqdm(val_raw.take(NUM_EVALUATE), total=NUM_EVALUATE):
    source_sentences.append(spa.numpy().decode("utf-8"))
    reference.append([eng.numpy().decode("utf-8")])

# パフォーマンスを向上させるため、単一バッチですべての機械翻訳を生成します。
candidate = decode_sequences(tf.constant(source_sentences))

# 結果を確認するためにいくつかのサンプルを出力
for i in range(5):
    print("-" * 20)
    print(f"Source:    {source_sentences[i]}")
    print(f"Reference: {reference[i][0]}")
    print(f"Machine:   {candidate[i]}")

### スコアの確認

In [ ]:
bleu = evaluate.load("bleu")
bleu_1 = bleu.compute(predictions=candidate, references=reference, max_order=1)
bleu_4 = bleu.compute(predictions=candidate, references=reference, max_order=4)

In [ ]:
bleu_1["bleu"]

In [ ]:
bleu_4["bleu"]

Copyright 2025 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License